# OMOP Condition Occurrence Table

Transforms FHIR Condition resources into OMOP CDM `condition_occurrence` table.

## Mapping: FHIR Condition → OMOP Condition_Occurrence

| OMOP Field | FHIR Source | Transformation |
|------------|-------------|----------------|
| condition_occurrence_id | Condition.id | Hash to integer |
| person_id | Condition.subject | Reference to person |
| condition_concept_id | Condition.code | Map SNOMED/ICD to OMOP concept |
| condition_start_date | Condition.onsetDateTime | Extract date |
| condition_start_datetime | Condition.onsetDateTime | Full timestamp |
| condition_end_date | Condition.abatementDateTime | Extract date |
| condition_end_datetime | Condition.abatementDateTime | Full timestamp |
| condition_type_concept_id | - | 32817 (EHR) |
| condition_status_concept_id | Condition.clinicalStatus | Map to OMOP status |
| visit_occurrence_id | Condition.encounter | Reference to visit |
| condition_source_value | Condition.code.coding[0].code | Original code |
| condition_source_concept_id | Condition.code | Source vocabulary concept |

## Vocabulary Mapping

| Source System | Target | Notes |
|--------------|--------|-------|
| SNOMED CT | Standard | Direct mapping |
| ICD-10-CM | Non-standard → SNOMED | Use concept_relationship |
| ICD-9-CM | Non-standard → SNOMED | Use concept_relationship |

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE silver_schema STRING DEFAULT 'bronze';
DECLARE OR REPLACE VARIABLE gold_schema STRING DEFAULT 'omop';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE silver_schema = COALESCE(:silver_schema, silver_schema);
SET VARIABLE gold_schema = COALESCE(:gold_schema, gold_schema);

USE IDENTIFIER(catalog_use || '.' || gold_schema);
SELECT current_catalog(), current_schema();

## Create Condition Occurrence Streaming Table

In [ ]:
DECLARE OR REPLACE VARIABLE create_condition_stmt STRING;

SET VARIABLE create_condition_stmt = "
CREATE OR REFRESH STREAMING TABLE condition_occurrence (
  -- Primary key
  condition_occurrence_id BIGINT NOT NULL COMMENT 'Unique condition occurrence identifier'
  
  -- Person reference
  ,person_id BIGINT NOT NULL COMMENT 'Reference to person table'
  
  -- Condition coding
  ,condition_concept_id INT NOT NULL COMMENT 'OMOP standard concept for condition'
  
  -- Dates
  ,condition_start_date DATE NOT NULL COMMENT 'Condition onset date'
  ,condition_start_datetime TIMESTAMP COMMENT 'Condition onset datetime'
  ,condition_end_date DATE COMMENT 'Condition resolution date'
  ,condition_end_datetime TIMESTAMP COMMENT 'Condition resolution datetime'
  
  -- Type and status
  ,condition_type_concept_id INT NOT NULL DEFAULT 32817 COMMENT 'Type: 32817=EHR'
  ,condition_status_concept_id INT DEFAULT 0 COMMENT 'Clinical status concept'
  ,stop_reason STRING COMMENT 'Reason condition ended'
  
  -- References
  ,provider_id BIGINT COMMENT 'Reference to provider table'
  ,visit_occurrence_id BIGINT COMMENT 'Reference to visit_occurrence table'
  ,visit_detail_id BIGINT COMMENT 'Reference to visit_detail table'
  
  -- Source values
  ,condition_source_value STRING COMMENT 'Original condition code'
  ,condition_source_concept_id INT DEFAULT 0 COMMENT 'Source vocabulary concept'
  ,condition_status_source_value STRING COMMENT 'Original clinical status'
  
  -- Code system info
  ,condition_code_system STRING COMMENT 'Source code system (SNOMED, ICD-10, etc.)'
  ,condition_display STRING COMMENT 'Display text for condition'
  
  -- Lineage
  ,fhir_condition_uuid STRING COMMENT 'Original FHIR Condition UUID'
  ,bundle_uuid STRING COMMENT 'Source bundle reference'
)
COMMENT 'OMOP CDM Condition Occurrence table - Diagnoses from FHIR Condition resources'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'gold'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS 
SELECT
  -- Generate integer condition_occurrence_id
  ABS(HASH(COALESCE(id::STRING, condition_uuid))) AS condition_occurrence_id
  
  -- Person reference
  ,ABS(HASH(
    COALESCE(
      REGEXP_EXTRACT(subject:reference::STRING, 'Patient/(.+)', 1),
      subject:reference::STRING
    )
  )) AS person_id
  
  -- Condition concept - placeholder using hash of code
  -- In production, join to OMOP vocabulary tables for proper concept_id
  ,COALESCE(
    ABS(HASH(code:coding[0]:code::STRING)) % 2000000000,  -- Placeholder
    0
  ) AS condition_concept_id
  
  -- Dates
  ,COALESCE(
    CAST(TRY_CAST(onsetDateTime::STRING AS TIMESTAMP) AS DATE),
    CAST(TRY_CAST(recordedDate::STRING AS TIMESTAMP) AS DATE),
    CURRENT_DATE()
  ) AS condition_start_date
  ,COALESCE(
    TRY_CAST(onsetDateTime::STRING AS TIMESTAMP),
    TRY_CAST(recordedDate::STRING AS TIMESTAMP)
  ) AS condition_start_datetime
  ,CAST(TRY_CAST(abatementDateTime::STRING AS TIMESTAMP) AS DATE) AS condition_end_date
  ,TRY_CAST(abatementDateTime::STRING AS TIMESTAMP) AS condition_end_datetime
  
  -- Type and status
  ,32817 AS condition_type_concept_id
  ,CASE clinicalStatus:coding[0]:code::STRING
    WHEN 'active' THEN 32902      -- Active
    WHEN 'resolved' THEN 32906    -- Resolved
    WHEN 'inactive' THEN 32904    -- Inactive
    WHEN 'remission' THEN 32905   -- In remission
    ELSE 0
  END AS condition_status_concept_id
  ,NULL AS stop_reason
  
  -- References
  ,NULL AS provider_id
  ,CASE 
    WHEN encounter:reference IS NOT NULL THEN
      ABS(HASH(REGEXP_EXTRACT(encounter:reference::STRING, 'Encounter/(.+)', 1)))
    ELSE NULL
  END AS visit_occurrence_id
  ,NULL AS visit_detail_id
  
  -- Source values
  ,code:coding[0]:code::STRING AS condition_source_value
  ,0 AS condition_source_concept_id
  ,clinicalStatus:coding[0]:code::STRING AS condition_status_source_value
  
  -- Code system info
  ,code:coding[0]:system::STRING AS condition_code_system
  ,COALESCE(code:coding[0]:display::STRING, code:text::STRING) AS condition_display
  
  -- Lineage
  ,condition_uuid AS fhir_condition_uuid
  ,bundle_uuid
  
FROM STREAM(" || catalog_use || "." || silver_schema || ".condition)
WHERE code IS NOT NULL
";

SELECT create_condition_stmt AS statement;

In [ ]:
EXECUTE IMMEDIATE create_condition_stmt;

In [ ]:
-- Verify condition_occurrence table
SELECT 
  condition_occurrence_id,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_source_value,
  condition_code_system,
  condition_display
FROM condition_occurrence
LIMIT 10;

In [ ]:
-- Conditions by code system
SELECT 
  CASE 
    WHEN condition_code_system LIKE '%snomed%' THEN 'SNOMED CT'
    WHEN condition_code_system LIKE '%icd-10%' THEN 'ICD-10'
    WHEN condition_code_system LIKE '%icd-9%' THEN 'ICD-9'
    ELSE condition_code_system
  END AS vocabulary,
  COUNT(*) AS count
FROM condition_occurrence
GROUP BY vocabulary
ORDER BY count DESC;

In [ ]:
-- Top conditions
SELECT 
  condition_source_value,
  condition_display,
  COUNT(*) AS occurrences
FROM condition_occurrence
GROUP BY condition_source_value, condition_display
ORDER BY occurrences DESC
LIMIT 20;